# Setup

In [1]:
import warnings
warnings.filterwarnings('ignore', message='BigQuery Storage module not found')

import tensorflow as tf
import numpy as np
import random
from google.cloud import bigquery

serve_fn = tf.saved_model.load('Format-Serving').signatures['serving_default']

with open('Format-Serving/assets/customer_id_vocab') as f:
    known_customers = [int(l.strip()) for l in f]
with open('Format-Serving/assets/product_id_vocab') as f:
    known_products = [int(l.strip()) for l in f]

GCP_PROJECT = 'b2b-recs'
BQ_TRAIN = f'{GCP_PROJECT}.raw_data.ternopil_train_v4'
BQ_TEST  = f'{GCP_PROJECT}.raw_data.ternopil_test_v4'
client = bigquery.Client(project=GCP_PROJECT)

# print(f'{len(known_customers):,} customers, {len(known_products):,} products')

# Random product

In [8]:
# Re-run to get a different customer/product
# Pick a random interaction from the TEST set (held-out last day)
from IPython.display import display, HTML

cust_str = ', '.join(str(c) for c in known_customers)

row = client.query(f"""
SELECT
    CAST(customer_id AS INT64) AS customer_id,
    cust_value,
    UNIX_SECONDS(date) AS date_unix,
    mge_main_cat_desc AS sub_category_v1,
    CAST(product_id AS INT64) AS product_id,
    art_name,
    brand_name AS brand,
    stratbuy_domain_desc AS category,
    mge_cat_desc AS sub_category_v2,
    days_since_last_purchase,
    cust_order_days_60d,
    cust_unique_products_60d
FROM `{BQ_TEST}`
WHERE customer_id IN ({cust_str}) AND cust_value > 0
ORDER BY RAND() LIMIT 1
""").to_dataframe().iloc[0]

cust_id = int(row['customer_id'])
cust_value = float(row['cust_value'])
date_unix = int(row['date_unix'])
input_subcat_v1 = row['sub_category_v1']
input_cat = row['category']
cust_last_purchase = int(row['days_since_last_purchase'] or 0)
cust_visits = int(row['cust_order_days_60d'] or 0)
cust_bought_SKU = int(row['cust_unique_products_60d'] or 0)
actual_product = int(row['product_id'])

display(HTML(f"""
<table style="border-collapse:collapse; font-size:14px;">
  <tr><th style="text-align:left; padding:4px 12px; border:1px solid #ccc;">customer_id</th>
      <td style="padding:4px 12px; border:1px solid #ccc;">{cust_id}</td></tr>
  <tr><th style="text-align:left; padding:4px 12px; border:1px solid #ccc;">date</th>
      <td style="padding:4px 12px; border:1px solid #ccc;">{date_unix}</td></tr>
  <tr><th style="text-align:left; padding:4px 12px; border:1px solid #ccc;">cust_value</th>
      <td style="padding:4px 12px; border:1px solid #ccc;">{cust_value:.0f}</td></tr>
  <tr><th style="text-align:left; padding:4px 12px; border:1px solid #ccc;">cust_last_purchase</th>
      <td style="padding:4px 12px; border:1px solid #ccc;">{cust_last_purchase}</td></tr>
  <tr><th style="text-align:left; padding:4px 12px; border:1px solid #ccc;">cust_visits</th>
      <td style="padding:4px 12px; border:1px solid #ccc;">{cust_visits}</td></tr>
  <tr><th style="text-align:left; padding:4px 12px; border:1px solid #ccc;">cust_bought_SKU</th>
      <td style="padding:4px 12px; border:1px solid #ccc;">{cust_bought_SKU}</td></tr>
  <tr><th style="text-align:left; padding:4px 12px; border:1px solid #ccc;">product</th>
      <td style="padding:4px 12px; border:1px solid #ccc;">{row['art_name']}</td></tr>
  <tr><th style="text-align:left; padding:4px 12px; border:1px solid #ccc;">brand</th>
      <td style="padding:4px 12px; border:1px solid #ccc;">{row['brand']}</td></tr>
  <tr><th style="text-align:left; padding:4px 12px; border:1px solid #ccc;">category</th>
      <td style="padding:4px 12px; border:1px solid #ccc;">{input_cat}</td></tr>
  <tr><th style="text-align:left; padding:4px 12px; border:1px solid #ccc;">sub_category</th>
      <td style="padding:4px 12px; border:1px solid #ccc;">{input_subcat_v1}</td></tr>
</table>
"""))

customer_id,120000616301
date,1717113600
cust_value,18166
cust_last_purchase,5
cust_visits,5
cust_bought_SKU,55
product,"NIKITA ГОРІЛКА КУКУРУДЗЯНА 0,7"
brand,NIKITA
category,SPIRITS
sub_category,ГОРІЛКА


# Top-20 recs

In [9]:
# Inference — payload: customer_id, date, cust_value, sub_category_v1, cust_last_purchase, cust_visits, cust_bought_SKU
from IPython.display import display, HTML

result = serve_fn(
    customer_id=tf.constant([cust_id], dtype=tf.int64),
    date=tf.constant([date_unix], dtype=tf.int64),
    cust_value=tf.constant([cust_value], dtype=tf.float32),
    sub_category_v1=tf.constant([input_subcat_v1], dtype=tf.string),
    cust_last_purchase=tf.constant([cust_last_purchase], dtype=tf.int64),
    cust_visits=tf.constant([cust_visits], dtype=tf.int64),
    cust_bought_SKU=tf.constant([cust_bought_SKU], dtype=tf.int64),
)
rec_ids = result['product_ids'].numpy()[0]
rec_scores = result['scores'].numpy()[0]

# Enrich from training data (broader product coverage) — include art_name
ids_str = ', '.join(str(int(p)) for p in rec_ids)
meta = client.query(f"""
SELECT DISTINCT
    CAST(product_id AS INT64) AS product_id,
    art_name,
    brand_name AS brand,
    stratbuy_domain_desc AS category,
    mge_main_cat_desc AS sub_category_v1,
    mge_cat_desc AS sub_category_v2
FROM `{BQ_TRAIN}`
WHERE product_id IN ({ids_str})
""").to_dataframe().set_index('product_id').to_dict('index')

# Build curated top-20: sub_category_v1 → category → top-150
TOP_N = 20
seen = {actual_product}  # exclude the input product from recommendations
curated = []  # list of (product_id, score)

for i in range(len(rec_ids)):
    pid = int(rec_ids[i])
    if pid in seen:
        continue
    m = meta.get(pid, {})
    if m.get('sub_category_v1') == input_subcat_v1:
        curated.append((pid, float(rec_scores[i])))
        seen.add(pid)

for i in range(len(rec_ids)):
    if len(curated) >= TOP_N:
        break
    pid = int(rec_ids[i])
    if pid in seen:
        continue
    m = meta.get(pid, {})
    if m.get('category') == input_cat:
        curated.append((pid, float(rec_scores[i])))
        seen.add(pid)

for i in range(len(rec_ids)):
    if len(curated) >= TOP_N:
        break
    pid = int(rec_ids[i])
    if pid in seen:
        continue
    curated.append((pid, float(rec_scores[i])))
    seen.add(pid)

curated = curated[:TOP_N]

hdr = "".join(f'<th style="padding:4px 10px; border:1px solid #ccc; background:#eee;">{c}</th>'
              for c in ['#', 'Product', 'Brand', 'Category', 'Sub-category', 'Sub-category 2', 'Product ID', 'Score'])

rows = []
for rank, (pid, score) in enumerate(curated, 1):
    m = meta.get(pid, {})
    cells = [
        f'{rank}',
        m.get('art_name', '?'),
        m.get('brand', '?'),
        m.get('category', '?'),
        m.get('sub_category_v1', '?'),
        m.get('sub_category_v2', '?'),
        str(pid),
        f'{score:.2f}',
    ]
    tds = "".join(f'<td style="padding:4px 10px; border:1px solid #ccc;">{v}</td>' for v in cells)
    rows.append(f'<tr>{tds}</tr>')

display(HTML(f"""
<table style="border-collapse:collapse; font-size:13px;">
  <tr>{hdr}</tr>
  {"\n  ".join(rows)}
</table>
"""))

#,Product,Brand,Category,Sub-category,Sub-category 2,Product ID,Score
1,"NIKITA ГОРІЛКА КУКУРУДЗЯНА 0,5",NIKITA,SPIRITS,ГОРІЛКА,ГОРІЛКА ЧИСТА,342060001001,462.18
2,"DISTIL №9 ГОРІЛКА 0,7",DISTIL #9,SPIRITS,ГОРІЛКА,ГОРІЛКА ЧИСТА,262443001001,460.80
3,"STOLI ГОРІЛКА 0,5",STOLI,SPIRITS,ГОРІЛКА,ГОРІЛКА ЧИСТА,342209001001,460.73
4,"HETMAN ГОРІЛКА 1,75",HETMAN,SPIRITS,ГОРІЛКА,ГОРІЛКА ЧИСТА,374354001001,460.40
5,"HETMAN ГОРІЛКА 0,7",HETMAN,SPIRITS,ГОРІЛКА,ГОРІЛКА ЧИСТА,324203001001,460.25
6,"FINLANDIA ГОР 0,7",FINLANDIA,SPIRITS,ГОРІЛКА,ГОРІЛКА ЧИСТА,201172001001,460.22
7,"NEMIROFF НАСТ МЕД З ПЕРЦ 0,5",NEMIROFF,SPIRITS,ГОРІЛКА,НАСТІЙКА,27998001003,459.76
8,"HETMAN ГОРІЛКА 0,5",HETMAN,SPIRITS,ГОРІЛКА,ГОРІЛКА ЧИСТА,324201001001,459.50
9,"ABSOLUT ГОРІЛКА 0,7",ABSOLUT,SPIRITS,ГОРІЛКА,ГОРІЛКА ЧИСТА,18846001001,459.08
10,"STOLI ГОРІЛКА 0,7",STOLI,SPIRITS,ГОРІЛКА,ГОРІЛКА ЧИСТА,342212001001,458.99
